In [34]:
# ============================================================
# SMARTPLAY AI RECOMMENDATION SYSTEM
# 06 - PERSONALIZED LEARNING PATH RECOMMENDATION
# ============================================================

import pandas as pd
import numpy as np
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [35]:
# ============================================================
# STEP 2 — LOAD DATA
# ============================================================

prediction_results = pd.read_csv(
    "../data/processed/cv_prediction_results.csv"
)

activity_clean = pd.read_csv(
    "../data/processed/activity_clean.csv"
)

assessment_clean = pd.read_csv(
    "../data/processed/assessment_clean.csv"
)

curriculum_clean = pd.read_csv(
    "../data/processed/curriculum_clean.csv"
)

print("Datasets loaded successfully.")

print("\nPrediction results:", prediction_results.shape)
print("Activity data:", activity_clean.shape)
print("Assessment data:", assessment_clean.shape)
print("Curriculum data:", curriculum_clean.shape)

Datasets loaded successfully.

Prediction results: (15, 5)
Activity data: (60, 12)
Assessment data: (30, 15)
Curriculum data: (20, 9)


In [36]:
# ============================================================
# STEP 3 — DATA VERIFICATION
# ============================================================

print("=" * 60)
print("DATA VERIFICATION")
print("=" * 60)

print("\nPrediction Results:")
display(prediction_results.head())

print("\nActivity Data:")
display(activity_clean.head())

print("\nAssessment Data:")
display(assessment_clean.head())

print("\nCurriculum Data:")
display(curriculum_clean.head())

DATA VERIFICATION

Prediction Results:


,Student_ID,Actual_Outcome,Logistic Regression_Prediction,Decision Tree_Prediction,Random Forest_Prediction
0,ST001,Successful,Successful,Successful,Successful
1,ST002,Successful,Needs Practice,Needs Practice,Needs Practice
2,ST003,Successful,Successful,Successful,Successful
3,ST004,Waiting,Needs Practice,Needs Practice,Needs Practice
4,ST005,Successful,Successful,Successful,Successful



Activity Data:


,Activity_ID,Student_ID,Lesson_ID,Activity_Type,Difficulty,Date,Completion_Status,Attempts,Time_Taken_Min,Score,Errors,Teacher_Evaluation
0,ACT001,ST001,LES001,Number Game,Easy,2026-01-10,Completed,1,8,95,None recorded,Excellent
1,ACT002,ST001,LES003,Phonics Exercise,Medium,2026-01-12,Completed,3,15,55,Letter sound confusion,Needs support
2,ACT003,ST002,LES001,Worksheet,Easy,2026-01-15,Completed,1,10,85,Minor mistakes,Good
3,ACT004,ST002,LES006,Pencil Tracing,Easy,2026-01-16,Completed,2,12,70,Uneven lines,Improving
4,ACT005,ST003,LES009,Creative Drawing,Medium,2026-01-17,Completed,1,18,92,None recorded,Excellent



Assessment Data:


,Assessment_ID,Student_ID,Subject,Assessment_Type,Pre_Test,Post_Test,Unit_Test,Monthly_Test,Term_Result,Competency_Score,Motor_Skills_Score,Creativity_Score,Interactive_Engagement,Learning_Progress,Recommended_Action
0,ASS001,ST001,Mathematics,Number Recognition,60,90,88,85,87,Number Recognition: 90%,75,70,90,30,Number puzzle activity
1,ASS002,ST001,English,Phonics,40,65,60,62,63,Phonics: 65%,70,75,85,25,Interactive phonics games
2,ASS003,ST002,Mathematics,Counting Objects,45,70,68,72,70,Counting: 70%,80,65,75,25,Counting with manipulatives
3,ASS004,ST002,Fine Motor Skills,Pencil Control,50,60,62,64,63,Pencil Control: 63%,60,65,70,10,Guided tracing exercises
4,ASS005,ST003,Creativity,Drawing Shapes,55,85,82,80,83,Shape Creation: 85%,80,90,88,30,Open-ended drawing



Curriculum Data:


,Lesson_ID,Grade,Subject,Topic,Competency,Learning_Objective,Difficulty_Level,Required_Skills,Classroom_Activities
0,LES021,Grade 1,Mathematics,Number Recognition,Number Sense,Identify numbers 11-50 correctly,Basic,Number sequence knowledge,Number chart activities
1,LES022,Grade 1,English,Simple Words,Reading Skills,Read and identify simple CVC words,Basic,Phonics knowledge,Word card exercises
2,LES023,Grade 1,Sinhala,Reading Practice,Language Skills,Read simple Sinhala words,Basic,Letter recognition,Reading aloud activities
3,LES024,Grade 1,Environmental Studies,School Environment,Social Awareness,Identify places and people in school,Beginner,Observation skills,School tour activity
4,LES025,Grade 1,Mathematics,Number Comparison,Logical Thinking,Compare greater and smaller numbers,Basic,Number recognition,Number comparison games


In [37]:
# ============================================================
# STEP 4 — CHECK CURRICULUM AND LESSON ALIGNMENT
# ============================================================

print("=" * 60)
print("CURRICULUM / LESSON ALIGNMENT CHECK")
print("=" * 60)

# Unique lesson IDs used in activity data
activity_lessons = set(activity_clean["Lesson_ID"].dropna().unique())

# Unique lesson IDs available in curriculum
curriculum_lessons = set(curriculum_clean["Lesson_ID"].dropna().unique())

print("\nActivity lesson IDs:")
print(sorted(activity_lessons))

print("\nCurriculum lesson IDs:")
print(sorted(curriculum_lessons))

# Lessons available in both datasets
common_lessons = activity_lessons.intersection(curriculum_lessons)

# Lessons used by students but missing from curriculum
missing_lessons = activity_lessons - curriculum_lessons

print("\nCommon lesson IDs:")
print(sorted(common_lessons))

print("\nLesson IDs used in activity data but NOT found in curriculum:")
print(sorted(missing_lessons))

print("\nNumber of common lessons:", len(common_lessons))
print("Number of missing lessons:", len(missing_lessons))

CURRICULUM / LESSON ALIGNMENT CHECK

Activity lesson IDs:
['LES001', 'LES002', 'LES003', 'LES004', 'LES005', 'LES006', 'LES009', 'LES010', 'LES011', 'LES012', 'LES013', 'LES014', 'LES015', 'LES016', 'LES017', 'LES018', 'LES019', 'LES021', 'LES022', 'LES023', 'LES024', 'LES025', 'LES026']

Curriculum lesson IDs:
['LES021', 'LES022', 'LES023', 'LES024', 'LES025', 'LES026', 'LES027', 'LES028', 'LES029', 'LES030', 'LES031', 'LES032', 'LES033', 'LES034', 'LES035', 'LES036', 'LES037', 'LES038', 'LES039', 'LES040']

Common lesson IDs:
['LES021', 'LES022', 'LES023', 'LES024', 'LES025', 'LES026']

Lesson IDs used in activity data but NOT found in curriculum:
['LES001', 'LES002', 'LES003', 'LES004', 'LES005', 'LES006', 'LES009', 'LES010', 'LES011', 'LES012', 'LES013', 'LES014', 'LES015', 'LES016', 'LES017', 'LES018', 'LES019']

Number of common lessons: 6
Number of missing lessons: 17


In [38]:
# ============================================================
# STEP 5 — CREATE SEMANTIC CURRICULUM ALIGNMENT
# ============================================================

import pandas as pd
import re

print("=" * 60)
print("SEMANTIC CURRICULUM ALIGNMENT")
print("=" * 60)


# ------------------------------------------------------------
# 1. Normalize text
# ------------------------------------------------------------

def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    # Remove special characters
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


# ------------------------------------------------------------
# 2. Create searchable curriculum text
# ------------------------------------------------------------

curriculum_match = curriculum_clean.copy()

text_columns = [
    "Subject",
    "Topic",
    "Competency",
    "Learning_Objective",
    "Required_Skills",
    "Classroom_Activities"
]

for col in text_columns:
    curriculum_match[col] = curriculum_match[col].fillna("").astype(str)

curriculum_match["Search_Text"] = (
    curriculum_match["Subject"] + " " +
    curriculum_match["Topic"] + " " +
    curriculum_match["Competency"] + " " +
    curriculum_match["Learning_Objective"] + " " +
    curriculum_match["Required_Skills"] + " " +
    curriculum_match["Classroom_Activities"]
).apply(normalize_text)


# ------------------------------------------------------------
# 3. Create assessment search text
# ------------------------------------------------------------

assessment_match = assessment_clean.copy()

assessment_match["Assessment_Text"] = (
    assessment_match["Subject"].fillna("").astype(str) + " " +
    assessment_match["Assessment_Type"].fillna("").astype(str) + " " +
    assessment_match["Competency_Score"].fillna("").astype(str)
).apply(normalize_text)


# ------------------------------------------------------------
# 4. Calculate keyword similarity
# ------------------------------------------------------------

def calculate_similarity(assessment_text, curriculum_text):
    
    assessment_words = set(assessment_text.split())
    curriculum_words = set(curriculum_text.split())
    
    if not assessment_words or not curriculum_words:
        return 0
    
    common_words = assessment_words.intersection(curriculum_words)
    
    return len(common_words)


# ------------------------------------------------------------
# 5. Match every assessment with curriculum lessons
# ------------------------------------------------------------

alignment_results = []

for _, assessment in assessment_match.iterrows():
    
    best_match = None
    best_score = -1
    
    assessment_subject = normalize_text(assessment["Subject"])
    assessment_text = assessment["Assessment_Text"]
    
    for _, lesson in curriculum_match.iterrows():
        
        curriculum_subject = normalize_text(lesson["Subject"])
        
        # Keyword similarity
        similarity = calculate_similarity(
            assessment_text,
            lesson["Search_Text"]
        )
        
        # Give strong priority to same subject
        if assessment_subject == curriculum_subject:
            similarity += 5
        
        if similarity > best_score:
            best_score = similarity
            best_match = lesson
    
    if best_match is not None:
        
        alignment_results.append({
            "Assessment_ID": assessment["Assessment_ID"],
            "Student_ID": assessment["Student_ID"],
            "Assessment_Subject": assessment["Subject"],
            "Assessment_Type": assessment["Assessment_Type"],
            "Term_Result": assessment["Term_Result"],
            "Competency_Score": assessment["Competency_Score"],
            "Motor_Skills_Score": assessment["Motor_Skills_Score"],
            "Creativity_Score": assessment["Creativity_Score"],
            "Interactive_Engagement": assessment["Interactive_Engagement"],
            "Learning_Progress": assessment["Learning_Progress"],
            
            "Recommended_Lesson_ID": best_match["Lesson_ID"],
            "Recommended_Subject": best_match["Subject"],
            "Recommended_Topic": best_match["Topic"],
            "Recommended_Competency": best_match["Competency"],
            "Learning_Objective": best_match["Learning_Objective"],
            "Difficulty_Level": best_match["Difficulty_Level"],
            "Required_Skills": best_match["Required_Skills"],
            "Classroom_Activities": best_match["Classroom_Activities"],
            
            "Alignment_Score": best_score
        })


alignment_results = pd.DataFrame(alignment_results)


# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("\nAlignment results shape:")
print(alignment_results.shape)

print("\nNumber of assessments aligned:")
print(alignment_results["Assessment_ID"].nunique())

print("\nSample curriculum alignment:")

display(
    alignment_results[
        [
            "Student_ID",
            "Assessment_Subject",
            "Assessment_Type",
            "Recommended_Lesson_ID",
            "Recommended_Topic",
            "Recommended_Competency",
            "Alignment_Score"
        ]
    ].head(10)
)

SEMANTIC CURRICULUM ALIGNMENT

Alignment results shape:
(30, 19)

Number of assessments aligned:
30

Sample curriculum alignment:


,Student_ID,Assessment_Subject,Assessment_Type,Recommended_Lesson_ID,Recommended_Topic,Recommended_Competency,Alignment_Score
0,ST001,Mathematics,Number Recognition,LES021,Number Recognition,Number Sense,8
1,ST001,English,Phonics,LES022,Simple Words,Reading Skills,7
2,ST002,Mathematics,Counting Objects,LES030,Addition Within 20,Calculation Skills,8
3,ST002,Fine Motor Skills,Pencil Control,LES027,Writing Letters,Writing Skills,3
4,ST003,Creativity,Drawing Shapes,LES034,Colouring,Creativity Skills,1
5,ST003,English,Letter Recognition,LES022,Simple Words,Reading Skills,6
6,ST004,Gross Motor Skills,Balance and Movement,LES022,Simple Words,Reading Skills,2
7,ST004,Mathematics,Shape Recognition,LES021,Number Recognition,Number Sense,7
8,ST005,Environmental Studies,Plants and Animals,LES029,Plants,Environmental Awareness,9
9,ST005,Creativity,Paper Craft,LES040,Craft Work,Creativity Skills,4


In [39]:
# ============================================================
# STEP 7 — IMPROVED SUBJECT-AWARE CURRICULUM ALIGNMENT
# ============================================================

import re
from difflib import SequenceMatcher

print("=" * 60)
print("IMPROVED SUBJECT-AWARE CURRICULUM ALIGNMENT")
print("=" * 60)


# ------------------------------------------------------------
# 1. Text normalization
# ------------------------------------------------------------

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def get_tokens(text):
    stop_words = {
        "the", "and", "of", "for", "to",
        "basic", "beginner", "simple",
        "skills", "skill", "knowledge",
        "activity", "activities",
        "practice"
    }

    return {
        word
        for word in normalize_text(text).split()
        if word not in stop_words
    }


# ------------------------------------------------------------
# 2. Prepare curriculum data
# ------------------------------------------------------------

curriculum_match = curriculum_clean.copy()

for col in [
    "Subject",
    "Topic",
    "Competency",
    "Learning_Objective",
    "Required_Skills",
    "Classroom_Activities"
]:
    curriculum_match[col] = (
        curriculum_match[col]
        .fillna("")
        .astype(str)
    )


# ------------------------------------------------------------
# 3. Prepare assessment data
# ------------------------------------------------------------

assessment_match = assessment_clean.copy()

for col in [
    "Subject",
    "Assessment_Type",
    "Competency_Score"
]:
    assessment_match[col] = (
        assessment_match[col]
        .fillna("")
        .astype(str)
    )


# ------------------------------------------------------------
# 4. Calculate curriculum matching score
# ------------------------------------------------------------

def calculate_curriculum_score(assessment, lesson):

    assessment_subject = normalize_text(
        assessment["Subject"]
    )

    curriculum_subject = normalize_text(
        lesson["Subject"]
    )

    assessment_type = normalize_text(
        assessment["Assessment_Type"]
    )

    topic = normalize_text(
        lesson["Topic"]
    )

    competency = normalize_text(
        lesson["Competency"]
    )

    learning_objective = normalize_text(
        lesson["Learning_Objective"]
    )

    required_skills = normalize_text(
        lesson["Required_Skills"]
    )

    classroom_activity = normalize_text(
        lesson["Classroom_Activities"]
    )


    score = 0


    # --------------------------------------------------------
    # SUBJECT MATCH
    # --------------------------------------------------------

    if assessment_subject == curriculum_subject:

        score += 100

    else:

        # Strong penalty for different subjects
        score -= 100


    # --------------------------------------------------------
    # EXACT ASSESSMENT TYPE → TOPIC
    # --------------------------------------------------------

    if assessment_type == topic:

        score += 100


    # --------------------------------------------------------
    # ASSESSMENT TYPE CONTAINED IN TOPIC
    # --------------------------------------------------------

    elif assessment_type in topic:

        score += 80


    # --------------------------------------------------------
    # TOPIC CONTAINED IN ASSESSMENT TYPE
    # --------------------------------------------------------

    elif topic in assessment_type:

        score += 60


    # --------------------------------------------------------
    # TOKEN OVERLAP
    # --------------------------------------------------------

    assessment_tokens = get_tokens(
        assessment_type
    )

    topic_tokens = get_tokens(
        topic
    )

    competency_tokens = get_tokens(
        competency
    )

    objective_tokens = get_tokens(
        learning_objective
    )

    required_tokens = get_tokens(
        required_skills
    )


    if len(assessment_tokens) > 0:

        topic_overlap = (
            len(assessment_tokens.intersection(topic_tokens))
            / len(assessment_tokens)
        )

        competency_overlap = (
            len(assessment_tokens.intersection(competency_tokens))
            / len(assessment_tokens)
        )

        objective_overlap = (
            len(assessment_tokens.intersection(objective_tokens))
            / len(assessment_tokens)
        )

        required_overlap = (
            len(assessment_tokens.intersection(required_tokens))
            / len(assessment_tokens)
        )

        score += topic_overlap * 50
        score += competency_overlap * 30
        score += objective_overlap * 20
        score += required_overlap * 15


    # --------------------------------------------------------
    # FUZZY TOPIC SIMILARITY
    # --------------------------------------------------------

    similarity = SequenceMatcher(
        None,
        assessment_type,
        topic
    ).ratio()

    score += similarity * 30


    return score


# ------------------------------------------------------------
# 5. Match every assessment to curriculum
# ------------------------------------------------------------

improved_alignment = []


for _, assessment in assessment_match.iterrows():

    assessment_subject = normalize_text(
        assessment["Subject"]
    )


    # --------------------------------------------------------
    # First: only same-subject curriculum lessons
    # --------------------------------------------------------

    same_subject_lessons = curriculum_match[
        curriculum_match["Subject"].apply(
            normalize_text
        ) == assessment_subject
    ]


    # --------------------------------------------------------
    # If same subject exists, ONLY use same subject
    # --------------------------------------------------------

    if len(same_subject_lessons) > 0:

        candidate_lessons = same_subject_lessons
        match_type = "Same Subject"


    # --------------------------------------------------------
    # Otherwise use all curriculum lessons
    # --------------------------------------------------------

    else:

        candidate_lessons = curriculum_match
        match_type = "Cross Subject Fallback"


    best_lesson = None
    best_score = float("-inf")


    # --------------------------------------------------------
    # Find best curriculum lesson
    # --------------------------------------------------------

    for _, lesson in candidate_lessons.iterrows():

        current_score = calculate_curriculum_score(
            assessment,
            lesson
        )

        if current_score > best_score:

            best_score = current_score
            best_lesson = lesson


    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    if best_lesson is not None:

        improved_alignment.append({

            "Assessment_ID":
                assessment["Assessment_ID"],

            "Student_ID":
                assessment["Student_ID"],

            "Assessment_Subject":
                assessment["Subject"],

            "Assessment_Type":
                assessment["Assessment_Type"],

            "Term_Result":
                assessment["Term_Result"],

            "Competency_Score":
                assessment["Competency_Score"],

            "Motor_Skills_Score":
                assessment["Motor_Skills_Score"],

            "Creativity_Score":
                assessment["Creativity_Score"],

            "Interactive_Engagement":
                assessment["Interactive_Engagement"],

            "Learning_Progress":
                assessment["Learning_Progress"],

            "Recommended_Lesson_ID":
                best_lesson["Lesson_ID"],

            "Recommended_Subject":
                best_lesson["Subject"],

            "Recommended_Topic":
                best_lesson["Topic"],

            "Recommended_Competency":
                best_lesson["Competency"],

            "Learning_Objective":
                best_lesson["Learning_Objective"],

            "Difficulty_Level":
                best_lesson["Difficulty_Level"],

            "Required_Skills":
                best_lesson["Required_Skills"],

            "Classroom_Activities":
                best_lesson["Classroom_Activities"],

            "Alignment_Score":
                round(best_score, 2),

            "Match_Type":
                match_type
        })


# ------------------------------------------------------------
# 6. Create final alignment dataframe
# ------------------------------------------------------------

alignment_results = pd.DataFrame(
    improved_alignment
)


# ------------------------------------------------------------
# 7. Verification
# ------------------------------------------------------------

print("\nAlignment results shape:")
print(alignment_results.shape)

print(
    "\nNumber of assessments aligned:",
    alignment_results["Assessment_ID"].nunique()
)

print(
    "Number of students:",
    alignment_results["Student_ID"].nunique()
)

print(
    "Number of recommended lessons:",
    alignment_results["Recommended_Lesson_ID"].nunique()
)


# ------------------------------------------------------------
# 8. Display alignment results
# ------------------------------------------------------------

print("\nImproved Curriculum Alignment:")

display(
    alignment_results[
        [
            "Student_ID",
            "Assessment_Subject",
            "Assessment_Type",
            "Recommended_Lesson_ID",
            "Recommended_Subject",
            "Recommended_Topic",
            "Recommended_Competency",
            "Alignment_Score",
            "Match_Type"
        ]
    ].head(30)
)

IMPROVED SUBJECT-AWARE CURRICULUM ALIGNMENT

Alignment results shape:
(30, 20)

Number of assessments aligned: 30
Number of students: 15
Number of recommended lessons: 16

Improved Curriculum Alignment:


,Student_ID,Assessment_Subject,Assessment_Type,Recommended_Lesson_ID,Recommended_Subject,Recommended_Topic,Recommended_Competency,Alignment_Score,Match_Type
0,ST001,Mathematics,Number Recognition,LES021,Mathematics,Number Recognition,Number Sense,302.50,Same Subject
1,ST001,English,Phonics,LES022,English,Simple Words,Reading Skills,124.47,Same Subject
2,ST002,Mathematics,Counting Objects,LES030,Mathematics,Addition Within 20,Calculation Skills,116.32,Same Subject
3,ST002,Fine Motor Skills,Pencil Control,LES027,Sinhala,Writing Letters,Writing Skills,-78.79,Cross Subject Fallback
4,ST003,Creativity,Drawing Shapes,LES023,Sinhala,Reading Practice,Language Skills,-86.00,Cross Subject Fallback
5,ST003,English,Letter Recognition,LES026,English,Sentence Formation,Writing Skills,113.33,Same Subject
6,ST004,Gross Motor Skills,Balance and Movement,LES032,Sinhala,Vocabulary Development,Language Development,-85.71,Cross Subject Fallback
7,ST004,Mathematics,Shape Recognition,LES021,Mathematics,Number Recognition,Number Sense,147.29,Same Subject
8,ST005,Environmental Studies,Plants and Animals,LES029,Environmental Studies,Plants,Environmental Awareness,210.00,Same Subject
9,ST005,Creativity,Paper Craft,LES040,Creative Activities,Craft Work,Creativity Skills,-60.71,Cross Subject Fallback


In [40]:
# ============================================================
# STEP 5C — DOMAIN-AWARE CURRICULUM ALIGNMENT
# ============================================================

print("=" * 60)
print("DOMAIN-AWARE CURRICULUM ALIGNMENT")
print("=" * 60)


# ------------------------------------------------------------
# 1. Map assessment areas to curriculum domains
# ------------------------------------------------------------

subject_domain_map = {

    "mathematics": [
        "mathematics"
    ],

    "english": [
        "english"
    ],

    "sinhala": [
        "sinhala"
    ],

    "environmental studies": [
        "environmental studies"
    ],

    "creativity": [
        "creative activities"
    ],

    "fine motor skills": [
        "creative activities",
        "life skills"
    ],

    "gross motor skills": [
        "life skills"
    ]
}


# ------------------------------------------------------------
# 2. Function to obtain curriculum domains
# ------------------------------------------------------------

def get_allowed_subjects(assessment_subject):

    subject = normalize_text(
        assessment_subject
    )

    return subject_domain_map.get(
        subject,
        [subject]
    )


# ------------------------------------------------------------
# 3. Create domain-aware recommendations
# ------------------------------------------------------------

domain_alignment = []


for _, assessment in assessment_match.iterrows():

    assessment_subject = normalize_text(
        assessment["Subject"]
    )

    allowed_subjects = get_allowed_subjects(
        assessment_subject
    )


    # --------------------------------------------------------
    # Find curriculum lessons belonging to allowed domains
    # --------------------------------------------------------

    candidate_lessons = curriculum_match[
        curriculum_match["Subject"].apply(
            normalize_text
        ).isin(allowed_subjects)
    ]


    # --------------------------------------------------------
    # If no curriculum lesson exists
    # --------------------------------------------------------

    if len(candidate_lessons) == 0:

        domain_alignment.append({

            "Assessment_ID":
                assessment["Assessment_ID"],

            "Student_ID":
                assessment["Student_ID"],

            "Assessment_Subject":
                assessment["Subject"],

            "Assessment_Type":
                assessment["Assessment_Type"],

            "Recommended_Lesson_ID":
                None,

            "Recommended_Subject":
                None,

            "Recommended_Topic":
                "No suitable curriculum lesson",

            "Recommended_Competency":
                "Teacher review required",

            "Learning_Objective":
                None,

            "Difficulty_Level":
                None,

            "Required_Skills":
                None,

            "Classroom_Activities":
                None,

            "Alignment_Score":
                0,

            "Match_Type":
                "No Suitable Curriculum Match"

        })

        continue


    # --------------------------------------------------------
    # Find best lesson within allowed domain
    # --------------------------------------------------------

    best_lesson = None

    best_score = float("-inf")


    for _, lesson in candidate_lessons.iterrows():

        current_score = calculate_curriculum_score(
            assessment,
            lesson
        )


        # Add additional preference for mapped domain
        current_score += 50


        if current_score > best_score:

            best_score = current_score

            best_lesson = lesson


    # --------------------------------------------------------
    # Store recommendation
    # --------------------------------------------------------

    domain_alignment.append({

        "Assessment_ID":
            assessment["Assessment_ID"],

        "Student_ID":
            assessment["Student_ID"],

        "Assessment_Subject":
            assessment["Subject"],

        "Assessment_Type":
            assessment["Assessment_Type"],

        "Recommended_Lesson_ID":
            best_lesson["Lesson_ID"],

        "Recommended_Subject":
            best_lesson["Subject"],

        "Recommended_Topic":
            best_lesson["Topic"],

        "Recommended_Competency":
            best_lesson["Competency"],

        "Learning_Objective":
            best_lesson["Learning_Objective"],

        "Difficulty_Level":
            best_lesson["Difficulty_Level"],

        "Required_Skills":
            best_lesson["Required_Skills"],

        "Classroom_Activities":
            best_lesson["Classroom_Activities"],

        "Alignment_Score":
            round(
                best_score,
                2
            ),

        "Match_Type":
            "Domain-Aware Match"

    })


# ------------------------------------------------------------
# 4. Create final dataframe
# ------------------------------------------------------------

alignment_results = pd.DataFrame(
    domain_alignment
)


# ------------------------------------------------------------
# 5. Verify results
# ------------------------------------------------------------

print("\nAlignment results shape:")
print(
    alignment_results.shape
)


print(
    "\nNumber of assessments:",
    alignment_results[
        "Assessment_ID"
    ].nunique()
)


print(
    "Number of students:",
    alignment_results[
        "Student_ID"
    ].nunique()
)


# Number of recommendation records
recommendation_records = (
    alignment_results[
        "Recommended_Lesson_ID"
    ]
    .notna()
    .sum()
)


print(
    "Number of recommendation records:",
    recommendation_records
)


# Number of unique recommended lessons
unique_recommended_lessons = (
    alignment_results[
        "Recommended_Lesson_ID"
    ]
    .dropna()
    .nunique()
)


print(
    "Number of unique recommended lessons:",
    unique_recommended_lessons
)


# ------------------------------------------------------------
# 6. Additional validation
# ------------------------------------------------------------

print("\nDomain-aware match types:")

print(
    alignment_results[
        "Match_Type"
    ].value_counts()
)


print(
    "\nMissing recommended lesson IDs:",
    alignment_results[
        "Recommended_Lesson_ID"
    ].isna().sum()
)


# ------------------------------------------------------------
# 7. Display final recommendations
# ------------------------------------------------------------

print("\nFINAL DOMAIN-AWARE RECOMMENDATIONS")


display(
    alignment_results[
        [
            "Student_ID",
            "Assessment_Subject",
            "Assessment_Type",
            "Recommended_Lesson_ID",
            "Recommended_Subject",
            "Recommended_Topic",
            "Recommended_Competency",
            "Alignment_Score",
            "Match_Type"
        ]
    ]
)

DOMAIN-AWARE CURRICULUM ALIGNMENT

Alignment results shape:
(30, 14)

Number of assessments: 30
Number of students: 15
Number of recommendation records: 30
Number of unique recommended lessons: 12

Domain-aware match types:
Match_Type
Domain-Aware Match    30
Name: count, dtype: int64

Missing recommended lesson IDs: 0

FINAL DOMAIN-AWARE RECOMMENDATIONS


,Student_ID,Assessment_Subject,Assessment_Type,Recommended_Lesson_ID,Recommended_Subject,Recommended_Topic,Recommended_Competency,Alignment_Score,Match_Type
0,ST001,Mathematics,Number Recognition,LES021,Mathematics,Number Recognition,Number Sense,352.50,Domain-Aware Match
1,ST001,English,Phonics,LES022,English,Simple Words,Reading Skills,174.47,Domain-Aware Match
2,ST002,Mathematics,Counting Objects,LES030,Mathematics,Addition Within 20,Calculation Skills,166.32,Domain-Aware Match
3,ST002,Fine Motor Skills,Pencil Control,LES040,Creative Activities,Craft Work,Creativity Skills,-40.00,Domain-Aware Match
4,ST003,Creativity,Drawing Shapes,LES034,Creative Activities,Colouring,Creativity Skills,-39.57,Domain-Aware Match
5,ST003,English,Letter Recognition,LES026,English,Sentence Formation,Writing Skills,163.33,Domain-Aware Match
6,ST004,Gross Motor Skills,Balance and Movement,LES039,Life Skills,Sharing and Caring,Social Skills,-37.37,Domain-Aware Match
7,ST004,Mathematics,Shape Recognition,LES021,Mathematics,Number Recognition,Number Sense,197.29,Domain-Aware Match
8,ST005,Environmental Studies,Plants and Animals,LES029,Environmental Studies,Plants,Environmental Awareness,260.00,Domain-Aware Match
9,ST005,Creativity,Paper Craft,LES040,Creative Activities,Craft Work,Creativity Skills,-10.71,Domain-Aware Match


In [43]:
# ============================================================
# STEP 5D — RECOMMENDATION QUALITY VALIDATION
# ============================================================

print("=" * 60)
print("RECOMMENDATION QUALITY VALIDATION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Make Alignment_Score numeric
# ------------------------------------------------------------

alignment_results["Alignment_Score"] = pd.to_numeric(
    alignment_results["Alignment_Score"],
    errors="coerce"
)


# ------------------------------------------------------------
# 2. Classify alignment quality
# ------------------------------------------------------------

def classify_alignment(score):

    if pd.isna(score):
        return "No Match"

    elif score >= 200:
        return "Excellent"

    elif score >= 100:
        return "Good"

    elif score >= 50:
        return "Moderate"

    else:
        return "Weak"


alignment_results["Alignment_Level"] = (
    alignment_results["Alignment_Score"]
    .apply(classify_alignment)
)


# ------------------------------------------------------------
# 3. Create recommendation status
# ------------------------------------------------------------

def recommendation_status(score):

    if pd.isna(score):
        return "Teacher Review Required"

    elif score >= 100:
        return "Recommended"

    else:
        return "Teacher Review Required"


alignment_results["Recommendation_Status"] = (
    alignment_results["Alignment_Score"]
    .apply(recommendation_status)
)


# ------------------------------------------------------------
# 4. Display quality summary
# ------------------------------------------------------------

print("\nAlignment Quality Summary:")
print(
    alignment_results["Alignment_Level"]
    .value_counts()
)


# ------------------------------------------------------------
# 5. Count recommendations
# ------------------------------------------------------------

recommended_count = (
    alignment_results["Recommendation_Status"]
    == "Recommended"
).sum()

review_count = (
    alignment_results["Recommendation_Status"]
    == "Teacher Review Required"
).sum()


print("\nRecommended:", recommended_count)

print(
    "Teacher review required:",
    review_count
)


# ------------------------------------------------------------
# 6. Unique recommended lessons
# ------------------------------------------------------------

print(
    "\nUnique recommended lessons:",
    alignment_results[
        "Recommended_Lesson_ID"
    ].nunique()
)


# ------------------------------------------------------------
# 7. Show weak recommendations
# ------------------------------------------------------------

print("\nWEAK / REVIEW REQUIRED RECOMMENDATIONS")

weak_recommendations = alignment_results[
    alignment_results["Recommendation_Status"]
    == "Teacher Review Required"
]


display(
    weak_recommendations[
        [
            "Student_ID",
            "Assessment_Subject",
            "Assessment_Type",
            "Recommended_Lesson_ID",
            "Recommended_Subject",
            "Recommended_Topic",
            "Recommended_Competency",
            "Alignment_Score",
            "Alignment_Level",
            "Recommendation_Status"
        ]
    ]
)

RECOMMENDATION QUALITY VALIDATION

Alignment Quality Summary:
Alignment_Level
Weak         13
Good         10
Excellent     7
Name: count, dtype: int64

Recommended: 17
Teacher review required: 13

Unique recommended lessons: 12

WEAK / REVIEW REQUIRED RECOMMENDATIONS


,Student_ID,Assessment_Subject,Assessment_Type,Recommended_Lesson_ID,Recommended_Subject,Recommended_Topic,Recommended_Competency,Alignment_Score,Alignment_Level,Recommendation_Status
3,ST002,Fine Motor Skills,Pencil Control,LES040,Creative Activities,Craft Work,Creativity Skills,-40.00,Weak,Teacher Review Required
4,ST003,Creativity,Drawing Shapes,LES034,Creative Activities,Colouring,Creativity Skills,-39.57,Weak,Teacher Review Required
6,ST004,Gross Motor Skills,Balance and Movement,LES039,Life Skills,Sharing and Caring,Social Skills,-37.37,Weak,Teacher Review Required
9,ST005,Creativity,Paper Craft,LES040,Creative Activities,Craft Work,Creativity Skills,-10.71,Weak,Teacher Review Required
12,ST007,Fine Motor Skills,Cutting and Pasting,LES039,Life Skills,Sharing and Caring,Social Skills,-30.54,Weak,Teacher Review Required
13,ST007,Creativity,Story Drawing,LES034,Creative Activities,Colouring,Creativity Skills,-36.36,Weak,Teacher Review Required
16,ST009,Gross Motor Skills,Throwing and Catching,LES039,Life Skills,Sharing and Caring,Social Skills,-26.92,Weak,Teacher Review Required
18,ST010,Creativity,Clay Modelling,LES034,Creative Activities,Colouring,Creativity Skills,-34.35,Weak,Teacher Review Required
21,ST011,Fine Motor Skills,Bead Threading,LES034,Creative Activities,Colouring,Creativity Skills,-39.57,Weak,Teacher Review Required
23,ST012,Creativity,Building Blocks,LES034,Creative Activities,Colouring,Creativity Skills,-40.00,Weak,Teacher Review Required


In [44]:
# ============================================================
# STEP 5E — BUILD PERSONALIZED STUDENT LEARNING PATHS
# ============================================================

print("=" * 60)
print("PERSONALIZED STUDENT LEARNING PATH GENERATION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Copy alignment results
# ------------------------------------------------------------

learning_path_source = alignment_results.copy()


# ------------------------------------------------------------
# 2. Add priority based on alignment quality
# ------------------------------------------------------------

def get_priority(row):

    score = row["Alignment_Score"]

    if pd.isna(score):
        return 0

    if score >= 200:
        return 3

    elif score >= 100:
        return 2

    else:
        return 1


learning_path_source["Priority"] = (
    learning_path_source.apply(
        get_priority,
        axis=1
    )
)


# ------------------------------------------------------------
# 3. Sort recommendations for each student
# ------------------------------------------------------------

learning_path_source = (
    learning_path_source
    .sort_values(
        by=[
            "Student_ID",
            "Priority",
            "Alignment_Score"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)


# ------------------------------------------------------------
# 4. Keep top 3 learning needs for each student
# ------------------------------------------------------------

learning_path_source["Path_Order"] = (
    learning_path_source
    .groupby("Student_ID")
    .cumcount() + 1
)


student_paths = learning_path_source[
    learning_path_source["Path_Order"] <= 3
].copy()


# ------------------------------------------------------------
# 5. Create learning-path stage
# ------------------------------------------------------------

def get_learning_stage(order):

    if order == 1:
        return "Immediate Support"

    elif order == 2:
        return "Skill Development"

    elif order == 3:
        return "Practice and Reinforcement"

    return "Additional Practice"


student_paths["Learning_Stage"] = (
    student_paths["Path_Order"]
    .apply(get_learning_stage)
)


# ------------------------------------------------------------
# 6. Select important columns
# ------------------------------------------------------------

student_paths = student_paths[
    [
        "Student_ID",
        "Path_Order",
        "Learning_Stage",
        "Assessment_ID",
        "Assessment_Subject",
        "Assessment_Type",
        "Recommended_Lesson_ID",
        "Recommended_Subject",
        "Recommended_Topic",
        "Recommended_Competency",
        "Learning_Objective",
        "Difficulty_Level",
        "Required_Skills",
        "Classroom_Activities",
        "Alignment_Score",
        "Alignment_Level",
        "Recommendation_Status"
    ]
]


# ------------------------------------------------------------
# 7. Verify
# ------------------------------------------------------------

print("\nNumber of students:")
print(student_paths["Student_ID"].nunique())


print("\nNumber of learning-path recommendations:")
print(len(student_paths))


print("\nRecommendations per student:")
print(
    student_paths
    .groupby("Student_ID")
    .size()
)


# ------------------------------------------------------------
# 8. Display personalized learning paths
# ------------------------------------------------------------

print("\nPERSONALIZED LEARNING PATHS")

display(
    student_paths
)

PERSONALIZED STUDENT LEARNING PATH GENERATION

Number of students:
15

Number of learning-path recommendations:
30

Recommendations per student:
Student_ID
ST001    2
ST002    2
ST003    2
ST004    2
ST005    2
ST006    2
ST007    2
ST008    2
ST009    2
ST010    2
ST011    2
ST012    2
ST013    2
ST014    2
ST015    2
dtype: int64

PERSONALIZED LEARNING PATHS


,Student_ID,Path_Order,Learning_Stage,Assessment_ID,Assessment_Subject,Assessment_Type,Recommended_Lesson_ID,Recommended_Subject,Recommended_Topic,Recommended_Competency,Learning_Objective,Difficulty_Level,Required_Skills,Classroom_Activities,Alignment_Score,Alignment_Level,Recommendation_Status
0,ST001,1,Immediate Support,ASS001,Mathematics,Number Recognition,LES021,Mathematics,Number Recognition,Number Sense,Identify numbers 11-50 correctly,Basic,Number sequence knowledge,Number chart activities,352.50,Excellent,Recommended
1,ST001,2,Skill Development,ASS002,English,Phonics,LES022,English,Simple Words,Reading Skills,Read and identify simple CVC words,Basic,Phonics knowledge,Word card exercises,174.47,Good,Recommended
2,ST002,1,Immediate Support,ASS003,Mathematics,Counting Objects,LES030,Mathematics,Addition Within 20,Calculation Skills,Solve simple addition problems up to 20,Basic,Counting ability,Addition using objects,166.32,Good,Recommended
3,ST002,2,Skill Development,ASS004,Fine Motor Skills,Pencil Control,LES040,Creative Activities,Craft Work,Creativity Skills,Create simple crafts using materials,Basic,Hand coordination,Paper craft activities,-40.00,Weak,Teacher Review Required
5,ST003,1,Immediate Support,ASS006,English,Letter Recognition,LES026,English,Sentence Formation,Writing Skills,Create simple sentences using words,Intermediate,Vocabulary knowledge,Sentence building cards,163.33,Good,Recommended
4,ST003,2,Skill Development,ASS005,Creativity,Drawing Shapes,LES034,Creative Activities,Colouring,Creativity Skills,Use colours appropriately in drawings,Beginner,Fine motor skills,Colouring worksheets,-39.57,Weak,Teacher Review Required
7,ST004,1,Immediate Support,ASS008,Mathematics,Shape Recognition,LES021,Mathematics,Number Recognition,Number Sense,Identify numbers 11-50 correctly,Basic,Number sequence knowledge,Number chart activities,197.29,Good,Recommended
6,ST004,2,Skill Development,ASS007,Gross Motor Skills,Balance and Movement,LES039,Life Skills,Sharing and Caring,Social Skills,Understand sharing and helping others,Beginner,Social interaction,Group activities,-37.37,Weak,Teacher Review Required
8,ST005,1,Immediate Support,ASS009,Environmental Studies,Plants and Animals,LES029,Environmental Studies,Plants,Environmental Awareness,Identify common plants around home and school,Beginner,Observation skills,Plant identification activity,260.00,Excellent,Recommended
9,ST005,2,Skill Development,ASS010,Creativity,Paper Craft,LES040,Creative Activities,Craft Work,Creativity Skills,Create simple crafts using materials,Basic,Hand coordination,Paper craft activities,-10.71,Weak,Teacher Review Required


In [45]:
# ============================================================
# STEP 6 — PERSONALIZED LEARNING PATH
# ============================================================

print("=" * 60)
print("STEP 6 — PERSONALIZED LEARNING PATH")
print("=" * 60)


# ------------------------------------------------------------
# 1. Calculate student activity performance
# ------------------------------------------------------------

activity_profile = (
    activity_clean
    .groupby("Student_ID")
    .agg(
        Activity_Count=("Activity_ID", "count"),
        Average_Activity_Score=("Score", "mean"),
        Average_Attempts=("Attempts", "mean"),
        Average_Time_Taken=("Time_Taken_Min", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 2. Calculate student assessment performance
# ------------------------------------------------------------

assessment_profile = (
    assessment_clean
    .groupby("Student_ID")
    .agg(
        Average_Pre_Test=("Pre_Test", "mean"),
        Average_Post_Test=("Post_Test", "mean"),
        Average_Term_Result=("Term_Result", "mean"),
        Average_Motor_Skills=("Motor_Skills_Score", "mean"),
        Average_Creativity=("Creativity_Score", "mean"),
        Average_Engagement=("Interactive_Engagement", "mean"),
        Average_Learning_Progress=("Learning_Progress", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 3. Combine student performance
# ------------------------------------------------------------

student_profile = pd.merge(
    activity_profile,
    assessment_profile,
    on="Student_ID",
    how="outer"
)


# ------------------------------------------------------------
# 4. Create overall performance score
# ------------------------------------------------------------

student_profile["Overall_Performance"] = (
    student_profile["Average_Activity_Score"].fillna(0) * 0.30
    +
    student_profile["Average_Term_Result"].fillna(0) * 0.30
    +
    student_profile["Average_Engagement"].fillna(0) * 0.15
    +
    student_profile["Average_Learning_Progress"].fillna(0) * 0.10
    +
    student_profile["Average_Motor_Skills"].fillna(0) * 0.075
    +
    student_profile["Average_Creativity"].fillna(0) * 0.075
)


# ------------------------------------------------------------
# 5. Classify student performance
# ------------------------------------------------------------

def classify_performance(score):

    if score >= 80:
        return "High"

    elif score >= 60:
        return "Moderate"

    else:
        return "Needs Support"


student_profile["Performance_Level"] = (
    student_profile["Overall_Performance"]
    .apply(classify_performance)
)


# ------------------------------------------------------------
# 6. Merge performance with learning paths
# ------------------------------------------------------------

personalized_paths = pd.merge(
    student_paths,
    student_profile,
    on="Student_ID",
    how="left"
)


# ------------------------------------------------------------
# 7. Determine teaching strategy
# ------------------------------------------------------------

def teaching_strategy(row):

    performance = row["Performance_Level"]
    alignment = row["Alignment_Level"]

    if alignment == "Weak":
        return "Teacher Review and Individual Support"

    if performance == "Needs Support":
        return "Guided Practice with Teacher Support"

    elif performance == "Moderate":
        return "Structured Practice and Reinforcement"

    else:
        return "Independent Practice with Extension"


personalized_paths["Teaching_Strategy"] = (
    personalized_paths.apply(
        teaching_strategy,
        axis=1
    )
)


# ------------------------------------------------------------
# 8. Determine recommended session duration
# ------------------------------------------------------------

def session_duration(row):

    difficulty = str(row["Difficulty_Level"]).lower()
    performance = row["Performance_Level"]

    if performance == "Needs Support":
        return "15–20 minutes"

    elif difficulty == "beginner":
        return "15 minutes"

    elif difficulty == "basic":
        return "20 minutes"

    elif difficulty == "intermediate":
        return "25 minutes"

    else:
        return "20 minutes"


personalized_paths["Recommended_Duration"] = (
    personalized_paths.apply(
        session_duration,
        axis=1
    )
)


# ------------------------------------------------------------
# 9. Determine teacher action
# ------------------------------------------------------------

def teacher_action(row):

    if row["Recommendation_Status"] == "Teacher Review Required":
        return (
            "Teacher should review the student's skill gap "
            "and adapt the activity before assigning."
        )

    elif row["Performance_Level"] == "Needs Support":
        return (
            "Provide guided demonstration, repeated practice, "
            "and individual feedback."
        )

    elif row["Performance_Level"] == "Moderate":
        return (
            "Provide structured practice and monitor progress."
        )

    else:
        return (
            "Allow independent practice and provide extension tasks."
        )


personalized_paths["Teacher_Action"] = (
    personalized_paths.apply(
        teacher_action,
        axis=1
    )
)


# ------------------------------------------------------------
# 10. Create final learning path columns
# ------------------------------------------------------------

final_learning_path = personalized_paths[
    [
        "Student_ID",
        "Path_Order",
        "Learning_Stage",
        "Assessment_ID",
        "Assessment_Subject",
        "Assessment_Type",
        "Recommended_Lesson_ID",
        "Recommended_Subject",
        "Recommended_Topic",
        "Recommended_Competency",
        "Learning_Objective",
        "Difficulty_Level",
        "Required_Skills",
        "Classroom_Activities",
        "Alignment_Score",
        "Alignment_Level",
        "Recommendation_Status",
        "Performance_Level",
        "Overall_Performance",
        "Teaching_Strategy",
        "Recommended_Duration",
        "Teacher_Action"
    ]
].copy()


# ------------------------------------------------------------
# 11. Round numerical values
# ------------------------------------------------------------

final_learning_path["Overall_Performance"] = (
    final_learning_path["Overall_Performance"]
    .round(2)
)

final_learning_path["Alignment_Score"] = (
    final_learning_path["Alignment_Score"]
    .round(2)
)


# ------------------------------------------------------------
# 12. Verify final learning paths
# ------------------------------------------------------------

print("\nNumber of students:")
print(
    final_learning_path["Student_ID"].nunique()
)

print("\nTotal learning-path records:")
print(
    len(final_learning_path)
)

print("\nLearning-path records per student:")
print(
    final_learning_path
    .groupby("Student_ID")
    .size()
)

print("\nPerformance levels:")
print(
    final_learning_path[
        [
            "Student_ID",
            "Performance_Level"
        ]
    ]
    .drop_duplicates()
    ["Performance_Level"]
    .value_counts()
)


# ------------------------------------------------------------
# 13. Display final personalized paths
# ------------------------------------------------------------

print("\nFINAL PERSONALIZED LEARNING PATH")

display(
    final_learning_path
)

STEP 6 — PERSONALIZED LEARNING PATH

Number of students:
15

Total learning-path records:
30

Learning-path records per student:
Student_ID
ST001    2
ST002    2
ST003    2
ST004    2
ST005    2
ST006    2
ST007    2
ST008    2
ST009    2
ST010    2
ST011    2
ST012    2
ST013    2
ST014    2
ST015    2
dtype: int64

Performance levels:
Performance_Level
Moderate         10
High              3
Needs Support     2
Name: count, dtype: int64

FINAL PERSONALIZED LEARNING PATH


,Student_ID,Path_Order,Learning_Stage,Assessment_ID,Assessment_Subject,Assessment_Type,Recommended_Lesson_ID,Recommended_Subject,Recommended_Topic,Recommended_Competency,...,Required_Skills,Classroom_Activities,Alignment_Score,Alignment_Level,Recommendation_Status,Performance_Level,Overall_Performance,Teaching_Strategy,Recommended_Duration,Teacher_Action
0,ST001,1,Immediate Support,ASS001,Mathematics,Number Recognition,LES021,Mathematics,Number Recognition,Number Sense,...,Number sequence knowledge,Number chart activities,352.50,Excellent,Recommended,Moderate,72.65,Structured Practice and Reinforcement,20 minutes,Provide structured practice and monitor progress.
1,ST001,2,Skill Development,ASS002,English,Phonics,LES022,English,Simple Words,Reading Skills,...,Phonics knowledge,Word card exercises,174.47,Good,Recommended,Moderate,72.65,Structured Practice and Reinforcement,20 minutes,Provide structured practice and monitor progress.
2,ST002,1,Immediate Support,ASS003,Mathematics,Counting Objects,LES030,Mathematics,Addition Within 20,Calculation Skills,...,Counting ability,Addition using objects,166.32,Good,Recommended,Moderate,66.32,Structured Practice and Reinforcement,20 minutes,Provide structured practice and monitor progress.
3,ST002,2,Skill Development,ASS004,Fine Motor Skills,Pencil Control,LES040,Creative Activities,Craft Work,Creativity Skills,...,Hand coordination,Paper craft activities,-40.00,Weak,Teacher Review Required,Moderate,66.32,Teacher Review and Individual Support,20 minutes,Teacher should review the student's skill gap ...
4,ST003,1,Immediate Support,ASS006,English,Letter Recognition,LES026,English,Sentence Formation,Writing Skills,...,Vocabulary knowledge,Sentence building cards,163.33,Good,Recommended,Moderate,79.54,Structured Practice and Reinforcement,25 minutes,Provide structured practice and monitor progress.
5,ST003,2,Skill Development,ASS005,Creativity,Drawing Shapes,LES034,Creative Activities,Colouring,Creativity Skills,...,Fine motor skills,Colouring worksheets,-39.57,Weak,Teacher Review Required,Moderate,79.54,Teacher Review and Individual Support,15 minutes,Teacher should review the student's skill gap ...
6,ST004,1,Immediate Support,ASS008,Mathematics,Shape Recognition,LES021,Mathematics,Number Recognition,Number Sense,...,Number sequence knowledge,Number chart activities,197.29,Good,Recommended,Moderate,60.72,Structured Practice and Reinforcement,20 minutes,Provide structured practice and monitor progress.
7,ST004,2,Skill Development,ASS007,Gross Motor Skills,Balance and Movement,LES039,Life Skills,Sharing and Caring,Social Skills,...,Social interaction,Group activities,-37.37,Weak,Teacher Review Required,Moderate,60.72,Teacher Review and Individual Support,15 minutes,Teacher should review the student's skill gap ...
8,ST005,1,Immediate Support,ASS009,Environmental Studies,Plants and Animals,LES029,Environmental Studies,Plants,Environmental Awareness,...,Observation skills,Plant identification activity,260.00,Excellent,Recommended,High,83.24,Independent Practice with Extension,15 minutes,Allow independent practice and provide extensi...
9,ST005,2,Skill Development,ASS010,Creativity,Paper Craft,LES040,Creative Activities,Craft Work,Creativity Skills,...,Hand coordination,Paper craft activities,-10.71,Weak,Teacher Review Required,High,83.24,Teacher Review and Individual Support,20 minutes,Teacher should review the student's skill gap ...


In [46]:
# ============================================================
# STEP 7 — SAVE FINAL PERSONALIZED LEARNING PATH
# ============================================================

import os

print("=" * 60)
print("STEP 7 — SAVE FINAL PERSONALIZED LEARNING PATH")
print("=" * 60)

# Create processed directory if it does not exist
os.makedirs("../data/processed", exist_ok=True)

# Save final learning path
output_path = (
    "../data/processed/"
    "final_personalized_learning_paths.csv"
)

final_learning_path.to_csv(
    output_path,
    index=False
)

print("\nFile saved successfully:")
print(output_path)

print("\nNumber of students:",
      final_learning_path["Student_ID"].nunique())

print("Number of learning-path records:",
      len(final_learning_path))

print("\nRecommendations per student:")

print(
    final_learning_path
    .groupby("Student_ID")
    .size()
)

STEP 7 — SAVE FINAL PERSONALIZED LEARNING PATH

File saved successfully:
../data/processed/final_personalized_learning_paths.csv

Number of students: 15
Number of learning-path records: 30

Recommendations per student:
Student_ID
ST001    2
ST002    2
ST003    2
ST004    2
ST005    2
ST006    2
ST007    2
ST008    2
ST009    2
ST010    2
ST011    2
ST012    2
ST013    2
ST014    2
ST015    2
dtype: int64


In [47]:
# ============================================================
# STEP 8 — STUDENT-WISE LEARNING PATH SUMMARY
# ============================================================

print("=" * 60)
print("STEP 8 — STUDENT-WISE LEARNING PATH SUMMARY")
print("=" * 60)

student_learning_summary = (
    final_learning_path
    .groupby("Student_ID")
    .agg(
        Number_of_Learning_Activities=(
            "Recommended_Lesson_ID",
            "count"
        ),

        Average_Alignment_Score=(
            "Alignment_Score",
            "mean"
        ),

        Performance_Level=(
            "Performance_Level",
            "first"
        ),

        Overall_Performance=(
            "Overall_Performance",
            "first"
        ),

        Recommended_Strategy=(
            "Teaching_Strategy",
            "first"
        )
    )
    .reset_index()
)

# Round numerical values
student_learning_summary["Average_Alignment_Score"] = (
    student_learning_summary["Average_Alignment_Score"]
    .round(2)
)

student_learning_summary["Overall_Performance"] = (
    student_learning_summary["Overall_Performance"]
    .round(2)
)

print("\nNumber of students:")
print(student_learning_summary["Student_ID"].nunique())

print("\nStudent-wise learning path summary:")

display(student_learning_summary)

STEP 8 — STUDENT-WISE LEARNING PATH SUMMARY

Number of students:
15

Student-wise learning path summary:


,Student_ID,Number_of_Learning_Activities,Average_Alignment_Score,Performance_Level,Overall_Performance,Recommended_Strategy
0,ST001,2,263.48,Moderate,72.65,Structured Practice and Reinforcement
1,ST002,2,63.16,Moderate,66.32,Structured Practice and Reinforcement
2,ST003,2,61.88,Moderate,79.54,Structured Practice and Reinforcement
3,ST004,2,79.96,Moderate,60.72,Structured Practice and Reinforcement
4,ST005,2,124.64,High,83.24,Independent Practice with Extension
5,ST006,2,243.88,Needs Support,58.85,Guided Practice with Teacher Support
6,ST007,2,-33.45,Moderate,63.26,Teacher Review and Individual Support
7,ST008,2,192.54,High,83.30,Independent Practice with Extension
8,ST009,2,161.54,Moderate,66.65,Structured Practice and Reinforcement
9,ST010,2,64.03,Moderate,76.60,Structured Practice and Reinforcement


In [48]:
# ============================================================
# STEP 9 — FINAL EXPORT
# ============================================================

import os

print("=" * 60)
print("STEP 9 — FINAL LEARNING PATH EXPORT")
print("=" * 60)

# Ensure output directory exists
os.makedirs("../data/processed", exist_ok=True)


# ------------------------------------------------------------
# 1. Save detailed personalized learning paths
# ------------------------------------------------------------

learning_path_file = (
    "../data/processed/"
    "final_personalized_learning_paths.csv"
)

final_learning_path.to_csv(
    learning_path_file,
    index=False
)


# ------------------------------------------------------------
# 2. Save student-wise summary
# ------------------------------------------------------------

summary_file = (
    "../data/processed/"
    "student_learning_path_summary.csv"
)

student_learning_summary.to_csv(
    summary_file,
    index=False
)


# ------------------------------------------------------------
# 3. Verify exported files
# ------------------------------------------------------------

print("\nExported Files:")
print("✓", learning_path_file)
print("✓", summary_file)


print("\nFinal Dataset Statistics:")

print(
    "Number of students:",
    final_learning_path["Student_ID"].nunique()
)

print(
    "Total learning-path records:",
    len(final_learning_path)
)

print(
    "Number of unique recommended lessons:",
    final_learning_path[
        "Recommended_Lesson_ID"
    ].nunique()
)

print(
    "Number of students in summary:",
    student_learning_summary[
        "Student_ID"
    ].nunique()
)


print("\n" + "=" * 60)
print("FINAL LEARNING PATH EXPORT COMPLETED")
print("=" * 60)

STEP 9 — FINAL LEARNING PATH EXPORT

Exported Files:
✓ ../data/processed/final_personalized_learning_paths.csv
✓ ../data/processed/student_learning_path_summary.csv

Final Dataset Statistics:
Number of students: 15
Total learning-path records: 30
Number of unique recommended lessons: 12
Number of students in summary: 15

FINAL LEARNING PATH EXPORT COMPLETED


In [49]:
# ============================================================
# STEP 10 — FINAL RECOMMENDATION REPORT
# ============================================================

print("=" * 60)
print("STEP 10 — FINAL PERSONALIZED LEARNING PATH REPORT")
print("=" * 60)


# ------------------------------------------------------------
# 1. Basic statistics
# ------------------------------------------------------------

number_of_students = (
    final_learning_path["Student_ID"].nunique()
)

number_of_records = len(final_learning_path)

number_of_lessons = (
    final_learning_path["Recommended_Lesson_ID"]
    .nunique()
)


print("\nBasic Statistics")
print("-" * 60)

print("Number of students:", number_of_students)
print("Total learning-path records:", number_of_records)
print("Unique recommended lessons:", number_of_lessons)


# ------------------------------------------------------------
# 2. Performance distribution
# ------------------------------------------------------------

print("\nPerformance Level Distribution")
print("-" * 60)

performance_distribution = (
    final_learning_path[
        ["Student_ID", "Performance_Level"]
    ]
    .drop_duplicates()
    ["Performance_Level"]
    .value_counts()
)

print(performance_distribution)


# ------------------------------------------------------------
# 3. Teaching strategy distribution
# ------------------------------------------------------------

print("\nTeaching Strategy Distribution")
print("-" * 60)

strategy_distribution = (
    final_learning_path[
        "Teaching_Strategy"
    ]
    .value_counts()
)

print(strategy_distribution)


# ------------------------------------------------------------
# 4. Recommendation status
# ------------------------------------------------------------

print("\nRecommendation Status Distribution")
print("-" * 60)

recommendation_distribution = (
    final_learning_path[
        "Recommendation_Status"
    ]
    .value_counts()
)

print(recommendation_distribution)


# ------------------------------------------------------------
# 5. Alignment quality
# ------------------------------------------------------------

print("\nAlignment Quality Distribution")
print("-" * 60)

alignment_distribution = (
    final_learning_path[
        "Alignment_Level"
    ]
    .value_counts()
)

print(alignment_distribution)


# ------------------------------------------------------------
# 6. Students requiring additional support
# ------------------------------------------------------------

print("\nStudents Requiring Additional Support")
print("-" * 60)

support_students = (
    student_learning_summary[
        student_learning_summary[
            "Performance_Level"
        ] == "Needs Support"
    ]
)

display(support_students)


# ------------------------------------------------------------
# 7. Final student-wise learning path
# ------------------------------------------------------------

print("\nFINAL STUDENT-WISE LEARNING PATH")
print("-" * 60)

display(
    final_learning_path[
        [
            "Student_ID",
            "Path_Order",
            "Recommended_Lesson_ID",
            "Recommended_Subject",
            "Recommended_Topic",
            "Recommended_Competency",
            "Difficulty_Level",
            "Classroom_Activities",
            "Performance_Level",
            "Teaching_Strategy",
            "Recommendation_Status"
        ]
    ]
)


# ------------------------------------------------------------
# 8. Completion message
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PERSONALIZED LEARNING PATH REPORT COMPLETED")
print("=" * 60)

STEP 10 — FINAL PERSONALIZED LEARNING PATH REPORT

Basic Statistics
------------------------------------------------------------
Number of students: 15
Total learning-path records: 30
Unique recommended lessons: 12

Performance Level Distribution
------------------------------------------------------------
Performance_Level
Moderate         10
High              3
Needs Support     2
Name: count, dtype: int64

Teaching Strategy Distribution
------------------------------------------------------------
Teaching_Strategy
Teacher Review and Individual Support    13
Structured Practice and Reinforcement    10
Independent Practice with Extension       4
Guided Practice with Teacher Support      3
Name: count, dtype: int64

Recommendation Status Distribution
------------------------------------------------------------
Recommendation_Status
Recommended                17
Teacher Review Required    13
Name: count, dtype: int64

Alignment Quality Distribution
--------------------------------------

,Student_ID,Number_of_Learning_Activities,Average_Alignment_Score,Performance_Level,Overall_Performance,Recommended_Strategy
5,ST006,2,243.88,Needs Support,58.85,Guided Practice with Teacher Support
10,ST011,2,79.18,Needs Support,54.78,Guided Practice with Teacher Support



FINAL STUDENT-WISE LEARNING PATH
------------------------------------------------------------


,Student_ID,Path_Order,Recommended_Lesson_ID,Recommended_Subject,Recommended_Topic,Recommended_Competency,Difficulty_Level,Classroom_Activities,Performance_Level,Teaching_Strategy,Recommendation_Status
0,ST001,1,LES021,Mathematics,Number Recognition,Number Sense,Basic,Number chart activities,Moderate,Structured Practice and Reinforcement,Recommended
1,ST001,2,LES022,English,Simple Words,Reading Skills,Basic,Word card exercises,Moderate,Structured Practice and Reinforcement,Recommended
2,ST002,1,LES030,Mathematics,Addition Within 20,Calculation Skills,Basic,Addition using objects,Moderate,Structured Practice and Reinforcement,Recommended
3,ST002,2,LES040,Creative Activities,Craft Work,Creativity Skills,Basic,Paper craft activities,Moderate,Teacher Review and Individual Support,Teacher Review Required
4,ST003,1,LES026,English,Sentence Formation,Writing Skills,Intermediate,Sentence building cards,Moderate,Structured Practice and Reinforcement,Recommended
5,ST003,2,LES034,Creative Activities,Colouring,Creativity Skills,Beginner,Colouring worksheets,Moderate,Teacher Review and Individual Support,Teacher Review Required
6,ST004,1,LES021,Mathematics,Number Recognition,Number Sense,Basic,Number chart activities,Moderate,Structured Practice and Reinforcement,Recommended
7,ST004,2,LES039,Life Skills,Sharing and Caring,Social Skills,Beginner,Group activities,Moderate,Teacher Review and Individual Support,Teacher Review Required
8,ST005,1,LES029,Environmental Studies,Plants,Environmental Awareness,Beginner,Plant identification activity,High,Independent Practice with Extension,Recommended
9,ST005,2,LES040,Creative Activities,Craft Work,Creativity Skills,Basic,Paper craft activities,High,Teacher Review and Individual Support,Teacher Review Required



PERSONALIZED LEARNING PATH REPORT COMPLETED
